# Scalar Locally-Hosted Media Downloader

**Purpose:** download every locally-hosted media file referenced by a Scalar book, so it can be re-uploaded into a destination install's own media folder.

**Why this exists:** the export/import notebook (`scalar_book_export_recreator.ipynb`) captures a book's text, structure, and annotations -- but it was never going to capture physical media files, because the RDF-JSON export format never contains them, only references to them (a filename, a path, sometimes a full URL). This notebook finds and downloads exactly the files that live on the source server's own `/media/` folder -- it deliberately EXCLUDES externally-hosted media (e.g. plates hosted on archive.org), since those aren't files you need to move; they're just external links that keep working wherever the book ends up.

**How it finds them:** it scans an already-exported book JSON (from the other notebook) for two kinds of reference:
1. Any field anywhere in the data whose value is a bare relative path starting with `media/` (thumbnails, backgrounds, banners, a media item's own file, etc.) -- these get reconstructed into full download URLs.
2. Any `href=`, `src=`, or `resource=` attribute inside a page's own HTML body that points at `/media/` on the source site -- this catches inline embeds and images referenced directly inside body text.

**How to use this notebook:**
1. Upload the JSON file exported by the other notebook (either the full `merged_graph.json` from its `crawl_state/` folder, or the final filtered export -- this tool works from either, and also re-fetches the book's own top-level scope directly as an extra safety net, since that's where a book's own background/thumbnail often live).
2. Set `BOOK_SLUG` below to match.
3. Run every cell top to bottom.
4. The final cell zips everything into one `media/` folder, ready to hand to your destination install's own upload process.


## 1. Setup

In [ ]:
import requests
import re
import json
import time
import shutil
from pathlib import Path
from urllib.parse import quote, unquote

# ---- CONFIGURE THIS ----
BOOK_SLUG = "piranesidigitalproject"
INSTALL_BASE = "https://scalar.usc.edu/works"
SOURCE_JSON_PATH = "merged_graph.json"   # the file you uploaded -- rename to match
# -------------------------

BOOK_URL = f"{INSTALL_BASE}/{BOOK_SLUG}"
DELAY_SECONDS = 0.5
MAX_RETRIES = 3
RETRY_DELAY_SECONDS = 5

OUTPUT_DIR = Path("media")
OUTPUT_DIR.mkdir(exist_ok=True)
STATE_DIR = Path("media_download_state")
STATE_DIR.mkdir(exist_ok=True)
DOWNLOADED_FILE = STATE_DIR / "downloaded.json"
FAILED_FILE = STATE_DIR / "failed.json"

def load_json(path, default):
    if path.exists():
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    return default

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=1, ensure_ascii=False)

with open(SOURCE_JSON_PATH, encoding="utf-8") as f:
    graph = json.load(f)

downloaded = set(load_json(DOWNLOADED_FILE, []))
failed = load_json(FAILED_FILE, [])

print(f"Loaded {len(graph)} nodes from {SOURCE_JSON_PATH}")
print(f"Already downloaded (from a previous run): {len(downloaded)}")

## 2. Also pull the book's own top-level scope
Belt-and-suspenders: the book's own background/thumbnail images live on the top-level Book node, which may or may not still be present in the file you uploaded depending on when it was exported relative to the other notebook's filtering step. This one small request makes sure those are covered either way.

In [ ]:
def _fetch(url):
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(url, timeout=30)
            if resp.status_code == 200:
                return json.loads(resp.text)
        except (requests.RequestException, json.JSONDecodeError):
            pass
        time.sleep(RETRY_DELAY_SECONDS)
    return None

book_scope = _fetch(f"{BOOK_URL}/rdf?format=json")
if book_scope:
    graph.update(book_scope)
    print(f"Merged book-level scope. Graph now has {len(graph)} nodes.")
else:
    print("Could not fetch book-level scope -- continuing with what was already loaded.")

## 3. Find every locally-hosted media reference

In [ ]:
def walk_values(obj):
    """Yield every string 'value' anywhere in the RDF-JSON structure."""
    if isinstance(obj, dict):
        if "value" in obj and isinstance(obj["value"], str):
            yield obj["value"]
        for v in obj.values():
            yield from walk_values(v)
    elif isinstance(obj, list):
        for item in obj:
            yield from walk_values(item)

def to_download_url(ref):
    """
    Normalize either a bare relative path ('media/Foo Bar.jpg') or an
    already-absolute URL on this install into one canonical download URL,
    URL-encoding only the parts that need it (spaces, etc.) without
    double-encoding an already-encoded URL.
    """
    if ref.startswith("http"):
        return ref  # already a full URL -- use as-is
    # bare relative path like 'media/Some File.jpg'
    encoded_path = quote(ref, safe="/")
    return f"{BOOK_URL}/{encoded_path}"

found_urls = set()

# Pass 1: any bare 'media/...' value anywhere in the structure
for val in walk_values(graph):
    if val.startswith("media/"):
        found_urls.add(to_download_url(val))

# Pass 2: href/src/resource attributes inside page HTML bodies that point at /media/
# on THIS install specifically (guards against accidentally sweeping in some other
# site's /media/ path that happens to appear in a hyperlink's text).
CONTENT_KEY = "http://rdfs.org/sioc/ns#content"
attr_pattern = re.compile(r'(?:href|src|resource)="([^"]*?/media/[^"]+)"')

for uri, node in graph.items():
    for content_val in node.get(CONTENT_KEY, []):
        html = content_val.get("value", "")
        if "/media/" not in html:
            continue
        for m in attr_pattern.finditer(html):
            ref = m.group(1)
            if ref.startswith("http") and BOOK_URL not in ref:
                continue  # absolute URL pointing at a DIFFERENT site -- not ours to fetch
            found_urls.add(to_download_url(ref))

print(f"Found {len(found_urls)} distinct locally-hosted media URLs to download.")

## 4. Download everything
Resumable the same way as the export notebook: anything already downloaded is skipped, and a failed file just stays pending for the next run rather than being silently marked done.

In [ ]:
pending = sorted(found_urls - downloaded)
print(f"{len(pending)} file(s) left to download.")

for i, url in enumerate(pending, start=1):
    filename = unquote(url.rsplit("/", 1)[-1])
    dest_path = OUTPUT_DIR / filename

    # avoid silently overwriting two different files that happen to share a name
    if dest_path.exists() and url not in downloaded:
        stem, suffix = dest_path.stem, dest_path.suffix
        n = 1
        while (OUTPUT_DIR / f"{stem}_{n}{suffix}").exists():
            n += 1
        dest_path = OUTPUT_DIR / f"{stem}_{n}{suffix}"

    print(f"[{i}/{len(pending)}] {filename}")

    success = False
    for attempt in range(MAX_RETRIES):
        try:
            resp = requests.get(url, timeout=60, stream=True)
            if resp.status_code == 200:
                with open(dest_path, "wb") as f:
                    for chunk in resp.iter_content(chunk_size=8192):
                        f.write(chunk)
                success = True
                break
            else:
                time.sleep(RETRY_DELAY_SECONDS)
        except requests.RequestException:
            time.sleep(RETRY_DELAY_SECONDS)

    if success:
        downloaded.add(url)
    else:
        print(f"  FAILED: {url}")
        if url not in failed:
            failed.append(url)

    save_json(DOWNLOADED_FILE, list(downloaded))
    save_json(FAILED_FILE, failed)
    time.sleep(DELAY_SECONDS)

print(f"\nDone. {len(downloaded)} total files downloaded, {len(failed)} still failing.")
if failed:
    print("Failed URLs (re-run this cell to retry):")
    for u in failed:
        print(" ", u)

## 5. Sanity check and zip for download

In [ ]:
files_on_disk = list(OUTPUT_DIR.glob("*"))
print(f"{len(files_on_disk)} file(s) currently in {OUTPUT_DIR}/")
total_size = sum(f.stat().st_size for f in files_on_disk)
print(f"Total size: {total_size / 1024 / 1024:.1f} MB")

zip_name = f"{BOOK_SLUG}_media"
shutil.make_archive(zip_name, "zip", OUTPUT_DIR)
print(f"Wrote {zip_name}.zip")

from google.colab import files
files.download(f"{zip_name}.zip")

## Notes for next steps

- Files are saved under their **original filenames**, matching exactly what the imported pages already reference as relative `media/filename` paths -- once these are uploaded into the destination install's own media directory, those references should resolve automatically with no link-rewriting needed.
- This deliberately does **not** download externally-hosted media (e.g. archive.org-hosted plates) -- those are just links, and they keep working wherever the book ends up.
- If `failed.json` (inside `media_download_state/`) isn't empty after a run, re-run Section 4 -- it only retries what's still missing.
- This is a different kind of gap than the absolute-URL link problem in the main migration guide's Step 9 -- that step is about hardcoded old-domain URLs baked into page text; this tool is about the physical files those (and other relative) references point at.